<img src="https://www.python.org/static/img/python-logo.png" width="300" alt="Python logo"  />
<font color='blue sky'>
$\Large\text{Pontificia Universidad Católica del Perú}$</font>

Pontificia Universidad Católica del Perú

$$\Large \textit{Proyecto para Análisis de Datos}$$

$$\large\textbf{Subastas Automotrices}$$

_Integrantes:_
* *Alejandra*
* *Davida Irayda Ponce Lacuaña*
* *Freddy Nina Apaza*
* *Oliver Malqui*
* *Martín Marlon Bendezu Palomino*
___

# 01_business
La adquisición de vehículos usados en subastas es una operación crítica que requiere decisiones rápidas. El problema principal es la incertidumbre sobre el estado real del activo. Actualmente, una proporción significativa de las compras (aproximadamente el 14%) resulta en una "mala compra" (IsBadBuy).
Impacto financiero:
* Costos de reparación: Unidades con fallas mecánicas ocultas que superan el presupuesto de reacondicionamiento.
* Depreciación de inventario: Vehículos que permanecen demasiado tiempo en stock por baja demanda o mala calidad percibida.
* Pérdida de margen: El costo de garantía (WarrantyCost) y el costo de adquisición (VehBCost) superan el precio de reventa final.

# Objetivo
El proyecto busca transformar los datos históricos de subastas en una ventaja competitiva mediante la construcción de un modelo de clasificación binaria. Este modelo actuará como un filtro previo a la compra, permitiendo:

** Identificar patrones de riesgo basados en el odómetro (VehOdo), la edad (VehicleAge) y el fabricante (Make).

** Comparar en tiempo real el precio de subasta vs. los valores de mercado de referencia (MMR).

## Listado de variables totales

IsBadBuy: Variable objetivo (1: mala compra, 0: buena).

PurchDate: Fecha de compra (Unix timestamp).

Auction: Casa de subasta.

VehYear: Año de fabricación.

VehicleAge: Antigüedad en años.

Make: Marca del vehículo.

Model: Modelo del vehículo.

Trim: Nivel de equipamiento/acabado.

SubModel: Variante específica del modelo.

Color: Color exterior.

Transmission: Tipo de transmisión (Auto/Manual).

WheelTypeID: ID del tipo de rueda.

WheelType: Descripción del tipo de rueda (Alloy, Covers, etc.).

VehOdo: Kilometraje (odómetro) al momento de compra.

Nationality: Origen de la marca (American, Asian, European).

Size: Tamaño del vehículo.

TopThreeAmericanName: Fabricante (GM, Ford, Chrysler u Other).

MMRAcquisitionAuctionAveragePrice: Precio promedio subasta (compra).

MMRAcquisitionAuctionCleanPrice: Precio subasta estado limpio (compra).

MMRAcquisitionRetailAveragePrice: Precio promedio venta público (compra).

MMRAcquisitonRetailCleanPrice: Precio venta público estado limpio (compra).

MMRCurrentAuctionAveragePrice: Precio promedio subasta (actual).

MMRCurrentAuctionCleanPrice: Precio subasta estado limpio (actual).

MMRCurrentRetailAveragePrice: Precio promedio venta público (actual).

MMRCurrentRetailCleanPrice: Precio venta público estado limpio (actual).

PRIMEUNIT: Indicador de unidad preferencial.

AUCGUART: Garantía de la subasta.

BYRNO: ID del comprador.

VNZIP1: Código postal del lugar de compra.

VNST: Estado/Provincia de la compra.

VehBCost: Costo de adquisición del vehículo.

IsOnlineSale: Indica si la venta fue en línea.

WarrantyCost: Costo de la garantía.

## Variable Objetivo

### La variable a predecir es: IsBadBuy
### Tipo: Binaria (0: Compra óptima / 1: Mala compra).

## Indicadores Clave

F1-Score Clase 1 (Balance): Equilibrio entre la detección de riesgo y la precisión, evitando rechazar demasiadas compras que podrían ser rentables.

ROC-AUC: Medida global de la capacidad del modelo para distinguir entre una buena y una mala compra.

## Estado actual

###El dataset presenta un desbalanceo (86% clase 0 / 14% clase 1), lo que requiere una estrategia técnica específica para no omitir los casos de riesgo.


# Prepocesamiento

LIBRERÍAS

In [3]:
!pip install plotly
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\PONCE\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [ ]:
# 1. Carga del dataset
# =========================================================
DATA_PATH = "../data/raw/06-kickAutomotriz.csv"

df = pd.read_csv(DATA_PATH)

print("Primeras filas del dataset:")
display(df.head())

In [ ]:
# Conversión del formato de fecha (YYYY-MM-DD)
df['PurchDate'] = pd.to_datetime(df['PurchDate'], unit='s')

# Visualización de los primeros registros
df.head()

In [ ]:
# Seleccionando la variable que nos muestra cuántos vehículos son malas compras (1) y cuántos no (0)
df.IsBadBuy.value_counts()

In [ ]:
# Creamos TARGET: 1 para compras exitosas, 0 para malas compras
df['TARGET'] = np.where(df['IsBadBuy'] == 0, 1, 0)

# Verificamos el conteo
df['TARGET'].value_counts()

In [ ]:
# Ahora veremos la proporción de compras exitosas (1) vs. compras riesgosas (0) en porcentaje
df["TARGET"].value_counts(normalize=True)

## 4.1. Calidad

### 4.1.1. Nulos

In [ ]:
# Eliminamos la columna original IsBadBuy para trabajar solo con TARGET
df.drop(columns=['IsBadBuy'], inplace=True)

# Verificamos que la columna ya no esté en el listado
df.columns

In [ ]:
# Veremos el porcentaje de valores nulos por columna (0.0 = 0% y 1.0 = 100%)
df.isnull().mean()

In [ ]:
# 1. Eliminar columna con alta tasa de vacíos
# En este dataset, PRIMEUNIT tiene casi un 96% de nulos.
df.drop(columns=['PRIMEUNIT'], inplace=True)

# 2. Reemplazo de vacíos con constantes
# Para GENERO (en este caso Color o Transmission si tuvieran nulos), usamos 'D'
df['Color'] = df['Color'].fillna('D')

# Para variables de uso o conteo, usamos 0
# Aplicamos esto a WheelTypeID que es una categoría numérica con nulos
df['WheelTypeID'] = df['WheelTypeID'].fillna(0)

# 3. Reemplazo de vacíos con la mediana (Para variables continuas/numéricas)
# Aplicamos a las columnas de precios MMR y WarrantyCost si tuvieran nulos
df['MMRAcquisitionAuctionAveragePrice'] = df['MMRAcquisitionAuctionAveragePrice'].fillna(df['MMRAcquisitionAuctionAveragePrice'].median())
df['WarrantyCost'] = df['WarrantyCost'].fillna(df['WarrantyCost'].median())

# Verificamos que ya no existan nulos en las columnas tratadas
df.isnull().sum()

### 4.1.2 Duplicados

In [ ]:
# 1. Homogenizar Transmisión (Evitar duplicados por escritura)
df['Transmission'] = df['Transmission'].str.upper()

# 2. Eliminar columnas con un solo valor (No aportan información)
# Si todos son 'FL' en VNST, no sirve para comparar.
if df['IsOnlineSale'].nunique() <= 1:
    df.drop(columns=['IsOnlineSale'], inplace=True)

# 3. Crear indicador de sobrecosto (KPI de Negocio)
df['Sobrecosto'] = df['VehBCost'] - df['MMRAcquisitionAuctionAveragePrice']

# 4. Limpieza final de nulos remanentes en WheelType
df['WheelType'] = df['WheelType'].fillna('Unknown')

In [ ]:
# Eliminación por alta tasa de vacíos
df.drop(columns=['AUCGUART'], inplace=True)

### 4.1.3 Tipos

In [ ]:
# Lista de categóricas para llenar con una constante
cols_categoricas = ['Trim', 'SubModel', 'Transmission', 'Nationality', 'Size', 'TopThreeAmericanName']
for col in cols_categoricas:
    df[col] = df[col].fillna('Unknown')

In [ ]:
# 1. Identificamos qué columnas de nuestra lista están realmente presentes en el df
cols_presentes = [col for col in cols_numericas if col in df.columns]

# 2. Aplicamos el llenado solo a esas columnas
for col in cols_presentes:
    df[col] = df[col].fillna(df[col].median())

# 3. Verificamos si alguna de la lista original quedó fuera
cols_faltantes = set(cols_numericas) - set(cols_presentes)
print(f"Columnas no encontradas: {cols_faltantes}")

In [ ]:
# 1. Limpiamos espacios invisibles en los nombres de las columnas
df.columns = df.columns.str.strip()

# 2. Definimos la lista con el nombre tal cual viene en el CSV (error ortográfico incluido)
cols_numericas = [
    'MMRAcquisitionAuctionAveragePrice',
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAquisitonRetailCleanPrice',  # Nombre real en el CSV (sin la 'i')
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice',
    'VehBCost',
    'Sobrecosto'
]

# 3. Llenamos con la mediana solo si la columna existe en el DataFrame
for col in cols_numericas:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())
    else:
        print(f"Advertencia: La columna '{col}' no se encontró.")

# 4. Verificación final de nulos en estas columnas
print(df[df.columns.intersection(cols_numericas)].isnull().sum())

In [ ]:
df.isnull().mean()

In [ ]:
df.describe()

In [ ]:
# 1. ELIMINACIÓN DE REGISTROS IMPOSIBLES (Ruido)
# Eliminamos filas donde el costo es 1 USD o el kilometraje es sospechosamente bajo
df = df[df['VehBCost'] > 100]
df = df[df['VehOdo'] > 100]

# 2. TRATAMIENTO DE CEROS EN PRECIOS (Imputación con Mediana)
# Identificamos columnas de precios que tienen ceros en lugar de nulos
cols_precios = [c for c in df.columns if 'MMR' in c]

for col in cols_precios:
    # Calculamos la mediana solo de los valores mayores a cero
    mediana_real = df[df[col] > 0][col].median()
    # Reemplazamos los ceros por esa mediana
    df[col] = df[col].replace(0, mediana_real)

# 3. RECALCULO DE VARIABLES DERIVADAS
# Actualizamos el Sobrecosto con los precios ya limpios
df['Sobrecosto'] = df['VehBCost'] - df['MMRAcquisitionAuctionAveragePrice']

# 4. VERIFICACIÓN FINAL
print(f"Registros restantes: {len(df)}")
print(f"Mínimo de VehBCost: {df.VehBCost.min()}")

In [ ]:
df.describe()

### 4.1.4 Limpieza de outliers y errores

In [ ]:
# Eliminamos registros con costos de adquisición demasiado bajos (errores de carga)
# Un vehículo de subasta por menos de 500 USD suele ser chatarra o error.
df = df[df['VehBCost'] > 500]

# Eliminamos registros donde el Sobrecosto sea una anomalía extrema
# (Pagar 18k menos que el mercado o 12k más sugiere datos corruptos)
df = df[(df['Sobrecosto'] > -5000) & (df['Sobrecosto'] < 8000)]

In [ ]:
# Columnas MMR a sanear
cols_mmr = [c for c in df.columns if 'MMR' in c]

for col in cols_mmr:
    # Reemplazamos valores sospechosos (0 o 1) por NaN para que fillna trabaje
    df.loc[df[col] <= 1, col] = None

    # Llenamos nulos con la mediana del año del vehículo (más preciso)
    df[col] = df.groupby('VehYear')[col].transform(lambda x: x.fillna(x.median()))

    # Si aún quedan nulos (años con pocos datos), usamos la mediana global
    df[col] = df[col].fillna(df[col].median())

# Recalculamos el Sobrecosto final con datos limpios
df['Sobrecosto'] = df['VehBCost'] - df['MMRAcquisitionAuctionAveragePrice']

In [ ]:
df.describe()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Lista de columnas de texto que el modelo debe entender
cols_categoricas = ['Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
                   'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName']

le = LabelEncoder()

for col in cols_categoricas:
    # Convertimos cada texto en un número único
    df[col] = le.fit_transform(df[col].astype(str))

# Verificamos que ahora todo el dataframe sea numérico
df.info()

## 4.2. Descriptivas

### 4.2.1 Descripcion

In [ ]:
df.describe()

### 4.2.2. Value counts

In [ ]:
# 1. Ver el desbalance del TARGET
print(df['TARGET'].value_counts(normalize=True))

# 2. Ver las marcas más frecuentes
print(df['Make'].value_counts().head(10))

# 3. Detectar categorías irrelevantes o "ruido"
print(df['Size'].value_counts())

## 4.2.3. Balanceo de clases

In [ ]:
# 1. Conteo absoluto y porcentual del TARGET
print("Conteo absoluto:")
print(df['TARGET'].value_counts())

print("\nConteo porcentual (Balance de clases):")
print(df['TARGET'].value_counts(normalize=True) * 100)

# 2. Visualización rápida para el reporte
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.countplot(x='TARGET', data=df, palette='viridis')
plt.title('Distribución de la Variable Objetivo (TARGET)')
plt.xticks([0, 1], ['Mala Compra (0)', 'Buena Compra (1)'])
plt.show()

## 4.3. Visualizaciones

### 4.3.1 Histogramas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de la figura
plt.figure(figsize=(10, 6))

# Histograma de Sobrecosto segmentado por TARGET
sns.histplot(data=df, x='Sobrecosto', hue='TARGET', kde=True, element="step", palette="viridis")

# Línea de referencia en 0 (donde el costo iguala al mercado)
plt.axvline(0, color='red', linestyle='--', label='Costo = Mercado')

plt.title('Análisis de Relevancia: Distribución de Sobrecosto por Clase', fontsize=14)
plt.xlabel('Sobrecosto (USD)', fontsize=12)
plt.ylabel('Frecuencia de Vehículos', fontsize=12)
plt.legend(title='Resultado', labels=['Buena Compra (1)', 'Mala Compra (0)', 'Referencia Mercado'])

plt.show()

### 4.3.2. Diagrama de Barras

In [ ]:
plt.figure(figsize=(14, 6))

# Graficamos la cantidad de vehículos por año, segmentado por TARGET
# Usamos un gráfico de barras apiladas o agrupadas
ax = sns.countplot(data=df, x='VehYear', hue='TARGET', palette='viridis')

# Personalización para que se vea profesional en tu reporte
plt.title('Relación entre el Año del Vehículo y el Riesgo de Compra', fontsize=14)
plt.xlabel('Año de Fabricación', fontsize=12)
plt.ylabel('Cantidad de Vehículos', fontsize=12)
plt.legend(title='Resultado', labels=['Mala Compra (0)', 'Buena Compra (1)'])
plt.xticks(rotation=45)

# Añadimos una cuadrícula ligera para facilitar la lectura
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

### 4.3.3. Mapa de calor

In [ ]:
# 1. Seleccionamos las variables numéricas clave para el negocio
cols_interes = [
    'VehicleAge', 'VehOdo', 'VehBCost', 
    'Sobrecosto', 'WarrantyCost', 'TARGET'
]

# 2. Calculamos la correlación de Pearson
corr = df[cols_interes].corr()

# 3. Visualizamos con un Mapa de Calor
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt=".2f")
plt.title('Matriz de Correlación: Identificando Factores de Riesgo', fontsize=14)
plt.show()

### 4.3.4. Target Vs Variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxplot para Edad del Vehículo
sns.boxplot(x='TARGET', y='VehicleAge', data=df, ax=axes[0], palette='coolwarm')
axes[0].set_title('Distribución de Edad vs TARGET')

# Boxplot para Sobrecosto
sns.boxplot(x='TARGET', y='Sobrecosto', data=df, ax=axes[1], palette='coolwarm')
axes[1].set_title('Distribución de Sobrecosto vs TARGET')

plt.show()